# Add a Update into the bronze cdc table

In [0]:
import random
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, LongType

# 1. Configuration
patient_ids = ["PT-1" ,"PT-3", "PT-2", "PT-158", "PT-318", "PT-671"]
selected_id = random.choice(patient_ids)
table_name = "patient_data.bronze_patients.bronze_patients_cdc"

schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("height_cm", LongType(), True),  
    StructField("age", LongType(), True),        
    StructField("sex", StringType(), True),
    StructField("op", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

try:
    # 3. Fetch latest record to get base data
    last_record_df = spark.table(table_name) \
        .filter(col("patient_id") == selected_id) \
        .orderBy(col("updated_at").desc()) \
        .limit(1)

    if last_record_df.count() == 0:
        print(f"Patient {selected_id} not found. Run your initial insert first.")
    else:
        # 4. Get data and modify weight
        patient_row = last_record_df.collect()[0]
        patient_data = patient_row.asDict()
        
        old_weight = patient_data['weight_kg']
        new_weight = round(old_weight + random.uniform(-2.0, 2.0), 1)
        
        patient_data['weight_kg'] = new_weight
        patient_data['op'] = "U" # Representing an Update in the log
        
        # 5. Create the new record DataFrame
        # We use the schema to ensure types match the Delta table
        new_record_df = spark.createDataFrame([patient_data], schema=schema) \
            .withColumn("updated_at", current_timestamp())

        # 6. Append to Delta table
        # We use .saveAsTable with mode("append")
        new_record_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(table_name)

        print(f"Append Successful! New row added for {selected_id}: {old_weight}kg -> {new_weight}kg")

except Exception as e:
    print(f"An error occurred: {e}")